# NBA Oracle Training — ALL ~6,500 features, single notebook, Colab T4

Self-contained: pulls all 8 feature files from HF dataset `LBJLincoln26/nba-feature-cache`, runs the patched engine.build(), trains XGBoost+LightGBM on ALL alive features (~3,800) plus TabICL on top-186, picks winner, archives + auto-promotes if Brier < production 0.22087.

Setup: Runtime → GPU T4. Tools → Secrets → `HF_TOKEN`. Then Run All. ~30-50min.

In [ ]:
!pip install -q tabicl xgboost lightgbm huggingface_hub scikit-learn==1.6.1 2>&1 | tail -5
import os, sys, json, time, pickle
from datetime import datetime, timezone
from pathlib import Path
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN
print('token len=', len(HF_TOKEN))

In [ ]:
from huggingface_hub import hf_hub_download
DATASET = 'LBJLincoln26/nba-feature-cache'
DATA_DIR = Path('/content/feature_data')
DATA_DIR.mkdir(exist_ok=True)
# 8 historical seasons + current = 11,532 games total. Multi-season unlocks
# time-series features (rolling 80-game windows, ts_acf, network centrality)
# that need long history. With only 2025-26 (1,257 games) ~3,800 alive;
# with full 8-season history ~4,400-4,500 alive.
SEASON_FILES = [f'games-{y}.json' for y in
                ['2017-18','2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25','2025-26']]
DATA_FILES = ['engine.py','referee_data.json','player_data_merged.json',
              'quarter_data.json','polymarket_data.json','full-odds-2025-26.json',
              'tracking_data.json']
FILES = DATA_FILES + SEASON_FILES
for f in FILES:
    p = hf_hub_download(repo_id=DATASET, filename=f'feature_data/{f}', repo_type='dataset', token=HF_TOKEN)
    target = DATA_DIR / f
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(p)
    print(f'  {f:35s} {Path(p).stat().st_size/1024/1024:6.2f} MB')

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('engine', str(DATA_DIR / 'engine.py'))
engine_mod = importlib.util.module_from_spec(spec); sys.modules['engine']=engine_mod; spec.loader.exec_module(engine_mod)
print('engine_version:', engine_mod.ENGINE_VERSION)

# Concatenate ALL 9 seasons (2017-18 … 2025-26 = 11,532 games)
games = []
for season_file in SEASON_FILES:
    raw = json.loads((DATA_DIR / season_file).read_text())
    season_games = raw.get('games', raw) if isinstance(raw, dict) else raw
    games.extend(season_games)
    print(f'  {season_file}: {len(season_games)} games')
print(f'TOTAL games: {len(games)}')

referee_data = json.loads((DATA_DIR / 'referee_data.json').read_text())
tracking_data = json.loads((DATA_DIR / 'tracking_data.json').read_text())

raw_pd = json.loads((DATA_DIR / 'player_data_merged.json').read_text())
player_data = {}
for k,v in raw_pd.items():
    if '|' in k: t,d = k.split('|',1); player_data[(t,d)]=v
    else: player_data[k]=v

raw_qd = json.loads((DATA_DIR / 'quarter_data.json').read_text())
quarter_data = {tuple(k.split('|',1)): v for k,v in raw_qd.items()}

raw_pm = json.loads((DATA_DIR / 'polymarket_data.json').read_text())
market_data = {}
for g in games:
    gid = g.get('game_id', '')
    d = (g.get('game_date') or '')[:10]
    h = (g.get('home',{}) or {}).get('team_abbr','') if isinstance(g.get('home'),dict) else ''
    a = (g.get('away',{}) or {}).get('team_abbr','') if isinstance(g.get('away'),dict) else ''
    key = f'{d}|{h}|{a}'
    if key in raw_pm: market_data[gid] = raw_pm[key]

# full-odds keys are "YYYY-MM-DD_AWAY@HOME"
raw_odds = json.loads((DATA_DIR / 'full-odds-2025-26.json').read_text())
odds_data = {}
for k, v in raw_odds.items():
    if not isinstance(v, dict) or '_' not in k or '@' not in k:
        continue
    try:
        date_part, matchup = k.split('_', 1)
        away, home = matchup.split('@', 1)
    except ValueError:
        continue
    base = v.get('base', {})
    odds_data[(date_part[:10], home, away)] = dict(base)

print(f'\nfeeds: ref={len(referee_data)} player={len(player_data)} quarter={len(quarter_data)} market={len(market_data)} odds={len(odds_data)} tracking={len(tracking_data)}')

In [ ]:
import numpy as np
engine = engine_mod.NBAFeatureEngine()
t0 = time.time()
X, y, feature_names = engine.build(games, referee_data=referee_data, player_data=player_data, quarter_data=quarter_data, market_data=market_data, odds_data=odds_data, tracking_data=tracking_data)
elapsed = time.time() - t0
X = np.nan_to_num(np.array(X, dtype=np.float32))
y = np.array(y, dtype=np.float64)
print(f'build: {X.shape[0]} games × {X.shape[1]} cols in {elapsed:.0f}s; y_mean={y.mean():.3f}')

variances = X.var(axis=0)
alive_mask = variances > 1e-10
alive_idx = np.where(alive_mask)[0]
print(f'alive: {len(alive_idx)}/{X.shape[1]} ({len(alive_idx)/X.shape[1]*100:.1f}%)')

X_full = X[:, alive_idx].astype(np.float32)
feat_names_full = [feature_names[i] for i in alive_idx]
ranked = sorted(alive_idx, key=lambda i: -variances[i])
top186_idx = ranked[:186]
X_186 = X[:, top186_idx].astype(np.float32)
feat_names_186 = [feature_names[i] for i in top186_idx]
print(f'X_full (XGB/LGBM): {X_full.shape}    X_186 (TabICL): {X_186.shape}')

In [ ]:
from sklearn.metrics import brier_score_loss
RANDOM_STATE = 1337; N_FOLDS = 10
EMBARGO = max(1, int(len(X_full) * 0.02))
FOLD_SIZE = len(X_full) // N_FOLDS
def cpcv_folds(n):
    for k in range(N_FOLDS):
        lo = k * FOLD_SIZE
        hi = lo + FOLD_SIZE if k < N_FOLDS - 1 else n
        m = np.ones(n, dtype=bool); m[max(0, lo - EMBARGO):min(n, hi + EMBARGO)] = False
        yield np.where(m)[0], np.arange(lo, hi)
all_results = []

In [ ]:
import xgboost as xgb
fold_briers, oof = [], np.zeros(len(X_full))
t0 = time.time()
for tr, te in cpcv_folds(len(X_full)):
    m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)
    m.fit(X_full[tr], y[tr])
    p = m.predict_proba(X_full[te])[:, 1]
    oof[te] = p
    fold_briers.append(float(brier_score_loss(y[te], p)))
xgb_brier = float(np.mean(fold_briers))
print(f'XGBoost (all {X_full.shape[1]} alive): Brier={xgb_brier:.5f} in {time.time()-t0:.0f}s')
all_results.append({'model':'xgboost','features':X_full.shape[1],'brier':xgb_brier,'fold_briers':fold_briers,'oof':oof.tolist()})

In [ ]:
import lightgbm as lgb
fold_briers, oof = [], np.zeros(len(X_full))
t0 = time.time()
for tr, te in cpcv_folds(len(X_full)):
    m = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1)
    m.fit(X_full[tr], y[tr])
    p = m.predict_proba(X_full[te])[:, 1]
    oof[te] = p
    fold_briers.append(float(brier_score_loss(y[te], p)))
lgb_brier = float(np.mean(fold_briers))
print(f'LightGBM (all {X_full.shape[1]} alive): Brier={lgb_brier:.5f} in {time.time()-t0:.0f}s')
all_results.append({'model':'lightgbm','features':X_full.shape[1],'brier':lgb_brier,'fold_briers':fold_briers,'oof':oof.tolist()})

In [ ]:
from tabicl import TabICLClassifier
best_tabicl = None
for ctx, temp in [(2048, 1.0), (3072, 1.0), (2048, 1.08)]:
    fold_briers, oof = [], np.zeros(len(X_186))
    t0 = time.time()
    for tr, te in cpcv_folds(len(X_186)):
        m = TabICLClassifier(n_estimators=1, softmax_temperature=temp, random_state=RANDOM_STATE)
        sub_tr = tr[-ctx:]
        m.fit(X_186[sub_tr], y[sub_tr])
        p = m.predict_proba(X_186[te])[:, 1]
        oof[te] = p
        fold_briers.append(float(brier_score_loss(y[te], p)))
    cv = float(np.mean(fold_briers))
    print(f'TabICL ctx={ctx} temp={temp}: Brier={cv:.5f} in {time.time()-t0:.0f}s')
    if best_tabicl is None or cv < best_tabicl['brier']:
        best_tabicl = {'model':'tabicl','features':186,'brier':cv,'ctx':ctx,'temp':temp,'fold_briers':fold_briers,'oof':oof.tolist()}
all_results.append(best_tabicl)
print(f"best TabICL: ctx={best_tabicl['ctx']} temp={best_tabicl['temp']} Brier={best_tabicl['brier']:.5f}")

In [ ]:
from sklearn.isotonic import IsotonicRegression
winner = min(all_results, key=lambda r: r['brier'])
print(f"\nWINNER: {winner['model']} on {winner['features']} features → Brier {winner['brier']:.5f}")
for r in all_results: print(f"  {r['model']:10s} {r['features']:>5d} feats  Brier {r['brier']:.5f}")

oof = np.array(winner['oof'])
iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y)
raw_b = brier_score_loss(y, oof); cal_b = brier_score_loss(y, iso.predict(oof))
print(f'isotonic: raw={raw_b:.5f} → calibrated={cal_b:.5f}')

if winner['model'] == 'xgboost':
    final = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)
    final.fit(X_full, y); final_features = feat_names_full; final_indices = list(map(int, alive_idx))
elif winner['model'] == 'lightgbm':
    final = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1)
    final.fit(X_full, y); final_features = feat_names_full; final_indices = list(map(int, alive_idx))
else:
    final = TabICLClassifier(n_estimators=1, softmax_temperature=winner['temp'], random_state=RANDOM_STATE)
    final.fit(X_186[-winner['ctx']:], y[-winner['ctx']:]); final_features = feat_names_186; final_indices = list(map(int, top186_idx))
print(f'final {winner["model"]} retrained on all {len(y)} games × {len(final_features)} features')

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
utc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')
bundle = {'model':final, 'calibrator':iso, 'feature_indices':final_indices, 'feature_names':final_features, 'cv_brier_mean':winner['brier'], 'cv_brier_per_fold':winner['fold_briers'], 'config':{'model_type':winner['model'],'n_features':len(final_features),'random_state':RANDOM_STATE,'cpcv_folds':N_FOLDS,'engine_version':engine_mod.ENGINE_VERSION,'features_alive_total':int(len(alive_idx)),'features_engine_total':int(X.shape[1]),'all_results_summary':[{'model':r['model'],'features':r['features'],'brier':r['brier']} for r in all_results]}, 'n_samples':int(X.shape[0]), 'trained_at':utc, 'trained_on':f'colab-multi-{winner["model"]}'}
if winner['model']=='tabicl': bundle['config']['ctx_size']=winner['ctx']; bundle['config']['softmax_temperature']=winner['temp']
PKL = f'/tmp/oracle-{winner["model"]}-{utc}.pkl'
with open(PKL, 'wb') as f: pickle.dump(bundle, f)
print(f'pkl: {PKL} ({Path(PKL).stat().st_size/1024/1024:.1f} MB)')

api = HfApi(token=HF_TOKEN)
api.create_repo('LBJLincoln26/nba-oracle-archive', repo_type='dataset', private=False, exist_ok=True)
api.upload_file(path_or_fileobj=PKL, path_in_repo=f'colab-multi-{winner["model"]}-{utc}.pkl', repo_id='LBJLincoln26/nba-oracle-archive', repo_type='dataset', commit_message=f'[colab-multi] {winner["model"]} Brier {winner["brier"]:.5f} alive={len(alive_idx)}/{X.shape[1]}')
print('archived')

try:
    cur = json.load(open(hf_hub_download(repo_id='LBJLincoln26/nba-oracle-model', filename='summary.json', repo_type='dataset', token=HF_TOKEN)))
    cur_brier = float(cur.get('cv_brier_mean', 0.99))
except Exception:
    cur_brier = 0.99
print(f'production: {cur_brier:.5f}')
if winner['brier'] < cur_brier:
    api.upload_file(path_or_fileobj=PKL, path_in_repo='nba-oracle.pkl', repo_id='LBJLincoln26/nba-oracle-model', repo_type='dataset', commit_message=f'[PROMOTE colab-multi] {winner["model"]} {winner["brier"]:.5f} (was {cur_brier:.5f})')
    summary = {k:v for k,v in bundle.items() if k not in ('model','calibrator')}
    summary['promoted_from_brier'] = cur_brier
    api.upload_file(path_or_fileobj=json.dumps(summary, indent=2, default=str).encode(), path_in_repo='summary.json', repo_id='LBJLincoln26/nba-oracle-model', repo_type='dataset', commit_message='[PROMOTE colab-multi] summary update')
    print(f'\n*** PROMOTED: {winner["model"]} {winner["brier"]:.5f} (was {cur_brier:.5f}) ***')
else:
    print(f'Not promoted: {winner["brier"]:.5f} >= {cur_brier:.5f}')